# 🚗 Automobile Market — Exploratory Data Analysis
**Dataset:** Automobile Dataset — vehicle characteristics including price, engine size, and fuel efficiency  
**Goal:** Uncover patterns in vehicle attributes and understand what drives car pricing  
**Author:** Arman Arabkhani | AUT Data Science


## 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('Car Dataset.csv')
print("Shape:", df.shape)
df.head()


## 2. Data Overview

In [ ]:
# Data types and structure
df.info()


In [ ]:
# Summary statistics
df.describe()


In [ ]:
# Duplicates
num_dup = df.duplicated().sum()
print(f"Duplicate rows: {num_dup}")
if num_dup > 0:
    df = df.drop_duplicates()
    print(f"Dropped {num_dup} duplicate(s). New shape: {df.shape}")


## 3. Handling Missing Values
The dataset uses `'?'` as a placeholder for missing values in some columns.
We replace them with NaN and handle each column appropriately.


In [ ]:
# Replace '?' with NaN across the entire dataframe
df.replace('?', np.nan, inplace=True)

# Convert numeric columns that may have been read as strings
numeric_cols_to_convert = ['price', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'num-of-doors']
for col in numeric_cols_to_convert:
    if col in df.columns and col != 'num-of-doors':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Summary of missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:\n", missing)


In [ ]:
# Fill missing numeric values with median (robust to outliers)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill missing categorical values with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after imputation:", df.isnull().sum().sum())


## 4. Outlier Detection & Removal
Using Z-score method (threshold = 3) to detect outliers across all numeric columns at once.
This corrects the original bug where only the last loop column was used for removal.


In [ ]:
analysis_cols = ['symboling', 'curb-weight', 'engine-size', 'city-mpg', 'highway-mpg', 'price']
analysis_cols = [c for c in analysis_cols if c in df.columns]

# ✅ Fixed: compute outlier mask for ALL columns simultaneously, remove any row flagged in any column
z_scores = np.abs((df[analysis_cols] - df[analysis_cols].mean()) / df[analysis_cols].std())
outlier_mask_any = (z_scores > 3).any(axis=1)

print(f"Outliers detected: {outlier_mask_any.sum()} rows out of {len(df)}")
df_clean = df[~outlier_mask_any].copy()
print(f"Clean dataset shape: {df_clean.shape}")


In [ ]:
# Visualise outliers for each column
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flatten(), analysis_cols):
    col_z = np.abs((df[col] - df[col].mean()) / df[col].std())
    is_outlier = col_z > 3
    ax.scatter(df.index[~is_outlier], df[col][~is_outlier],
               marker='o', color='steelblue', alpha=0.5, label='Normal', s=15)
    ax.scatter(df.index[is_outlier], df[col][is_outlier],
               marker='x', color='red', label='Outlier', s=40)
    ax.set_title(f'{col}')
    ax.set_xlabel('Index')
    ax.set_ylabel(col)
    ax.legend(fontsize=7)

plt.suptitle('Outlier Detection via Z-Score (threshold = 3)', fontsize=14)
plt.tight_layout()
plt.show()


## 5. Distributions of Numeric Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), analysis_cols):
    ax.hist(df_clean[col].dropna(), bins=25, color='steelblue', edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.suptitle('Numeric Feature Distributions (After Outlier Removal)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Combined boxplot for a quick spread comparison
plt.figure(figsize=(10, 5))
df_clean[['curb-weight', 'engine-size', 'city-mpg', 'highway-mpg']].boxplot()
plt.title('Boxplot — Key Numeric Features')
plt.ylabel('Value')
plt.tight_layout()
plt.show()


## 6. Categorical Feature Distributions

In [ ]:
categorical_columns = ['fuel-type', 'aspiration', 'body-style', 'drive-wheels', 'engine-location']
categorical_columns = [c for c in categorical_columns if c in df_clean.columns]

fig, axes = plt.subplots(1, len(categorical_columns), figsize=(18, 4))
for ax, col in zip(axes, categorical_columns):
    order = df_clean[col].value_counts().index
    sns.countplot(data=df_clean, x=col, order=order, ax=ax, palette='Set2')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()


## 7. Correlation Analysis
Examining relationships between key numeric features.


In [ ]:
corr_cols = ['curb-weight', 'engine-size', 'city-mpg', 'highway-mpg', 'price', 'symboling']
corr_cols = [c for c in corr_cols if c in df_clean.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap — Key Numeric Features')
plt.tight_layout()
plt.show()

print("""
Key observations:
- engine-size and curb-weight are strongly positively correlated → heavier cars have bigger engines
- city-mpg and highway-mpg are strongly correlated with each other (as expected)
- engine-size and city-mpg are strongly negatively correlated → bigger engines = worse fuel economy
- price correlates positively with engine-size and curb-weight
""")


## 8. Price Analysis

In [ ]:
# Price distribution
plt.figure(figsize=(8, 4))
df_clean['price'].dropna().hist(bins=30, color='coral', edgecolor='white')
plt.title('Price Distribution')
plt.xlabel('Price (USD)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Mean price:   ${df_clean['price'].mean():,.0f}")
print(f"Median price: ${df_clean['price'].median():,.0f}")
print(f"Price range:  ${df_clean['price'].min():,.0f} — ${df_clean['price'].max():,.0f}")


In [ ]:
# Price by body style
plt.figure(figsize=(10, 5))
sns.boxplot(x='body-style', y='price', data=df_clean, palette='Set3')
plt.title('Price Distribution by Body Style')
plt.xlabel('Body Style')
plt.ylabel('Price (USD)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print("\nMedian price by body style:")
print(df_clean.groupby('body-style')['price'].median().sort_values(ascending=False))


In [ ]:
# Price by drive wheels and body style
plt.figure(figsize=(12, 5))
sns.boxplot(x='drive-wheels', y='price', hue='body-style', data=df_clean, palette='Set3')
plt.title('Price by Drive Wheels & Body Style')
plt.xlabel('Drive Wheels')
plt.ylabel('Price (USD)')
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Mean & Median price by manufacturer
price_stats = df_clean.groupby('make')['price'].agg(['mean', 'median']).reset_index()
price_stats = price_stats.sort_values('mean', ascending=False)

plt.figure(figsize=(14, 6))
x = np.arange(len(price_stats))
width = 0.4
plt.bar(x - width/2, price_stats['mean'],   width, label='Mean Price',   color='steelblue', alpha=0.85)
plt.bar(x + width/2, price_stats['median'], width, label='Median Price', color='coral',     alpha=0.85)
plt.xticks(x, price_stats['make'], rotation=90)
plt.xlabel('Manufacturer')
plt.ylabel('Price (USD)')
plt.title('Mean & Median Prices by Manufacturer')
plt.legend()
plt.tight_layout()
plt.show()


## 9. Engine Size vs. Fuel Efficiency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mpg_col, title in zip(axes, ['city-mpg', 'highway-mpg'],
                               ['Engine Size vs. City MPG', 'Engine Size vs. Highway MPG']):
    ax.scatter(df_clean['engine-size'], df_clean[mpg_col],
               alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
    # Add trend line
    m, b = np.polyfit(df_clean['engine-size'].dropna(), df_clean[mpg_col].dropna(), 1)
    x_line = np.linspace(df_clean['engine-size'].min(), df_clean['engine-size'].max(), 100)
    ax.plot(x_line, m * x_line + b, 'r--', linewidth=1.5, label='Trend')
    ax.set_xlabel('Engine Size')
    ax.set_ylabel(mpg_col)
    ax.set_title(title)
    ax.legend()

plt.suptitle('Engine Size vs. Fuel Efficiency', fontsize=14)
plt.tight_layout()
plt.show()

print("Correlation — engine-size vs city-mpg:   ", round(df_clean['engine-size'].corr(df_clean['city-mpg']), 3))
print("Correlation — engine-size vs highway-mpg:", round(df_clean['engine-size'].corr(df_clean['highway-mpg']), 3))


## 10. Summary & Conclusions

### Key Findings

**Engine & Weight:**
- Engine size and curb weight are strongly correlated — heavier vehicles consistently have larger engines.
- Outliers in these features typically represent luxury or high-performance models.

**Fuel Efficiency:**
- Larger engine size is strongly negatively correlated with both city and highway MPG, confirming the power-vs-economy trade-off.
- Highway MPG is consistently higher than city MPG across all vehicle types, as expected.

**Pricing:**
- Hardtop and convertible body styles command the highest median prices.
- Rear-wheel-drive vehicles tend to be more expensive, likely due to sports/luxury association.
- There is significant variation in price within manufacturers — outliers often represent premium trim levels.

### 🔧 Next Steps
- Build a **regression model** to predict car price from features such as engine size, curb weight, and body style
- Apply feature engineering (e.g., power-to-weight ratio)
- Deploy findings as an interactive Streamlit dashboard
